# 03 — Glossary workflows trigger

For governance domains whose contracts changed since the last run, trigger
the `Create-Glossary-Term` and `Asset-Curation-Update` workflows so terms /
descriptions are routed to the approver in the Purview portal.

This notebook is optional — the Streamlit Glossary editor triggers workflows
on-demand. Use this scheduled run only for batch sync from a `contracts/`
folder (mirrors the pattern in `fabric-sdlc-governance/contracts/`).

In [ ]:
import sys, pathlib, json
REPO_ROOT = pathlib.Path("/lakehouse/default/Files/repo/fabric-lineage-graph")
if REPO_ROOT.exists():
    sys.path.insert(0, str(REPO_ROOT))

from api.udf_glossary import create_term
from api.udf_workflows import trigger_bulk_update

contracts = pathlib.Path("/lakehouse/default/Files/repo/fabric-sdlc-governance/contracts/dictionary")
new_terms = list(contracts.glob("*.json")) if contracts.exists() else []
print(f"Found {len(new_terms)} dictionary files")

In [ ]:
for f in new_terms:
    spec = json.loads(f.read_text())
    try:
        res = create_term(spec["name"], spec["definition"], spec.get("steward", ""))
        print(f"  {spec['name']:30} -> guid={res.get('guid','?')}")
    except Exception as exc:
        print(f"  {spec['name']:30} FAILED — {exc!r}")